# EVALUACIÓN DEL MODELO RAG


- Chunking por reseña completa vs por oración
- Generación y caché de embeddings con `multilingual-e5-small`
- Indexación con FAISS
- Búsqueda semántica
- Generación de respuestas con Gemini

In [1]:
import sys
sys.path.append('../modelos')

import pandas as pd
from src.modelos.rag_utils import (
    cargar_modelo_embeddings,
    chunking_por_resena,
    chunking_por_oracion,
    generar_embeddings,
    construir_indice_faiss,
    guardar_chunks,
    buscar_chunks,
    configurar_gemini,
    pipeline_rag
)

C:\Users\eveca\OneDrive\Escritorio\Chat TurisPOS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Cargar el corpus

In [2]:
corpus = pd.read_csv('../../data/processed/corpus_final.csv')

print(f'Total reseñas: {len(corpus)}')
print(f'Columnas: {corpus.columns.tolist()}')
corpus.head(3)

Total reseñas: 5063
Columnas: ['lugar', 'texto', 'calificacion', 'categoria', 'fuente', 'idioma_comentario', 'comentarios_espanol', 'emojis', 'polaridad']


,lugar,texto,calificacion,categoria,fuente,idioma_comentario,comentarios_espanol,emojis,polaridad
0,Playa Manuel Antonio,Manuel Antonio Beach is absolutely stunning. T...,5.0,parque,google_maps,en,La playa de Manuel Antonio es absolutamente im...,NaN,positivo
1,Playa Manuel Antonio,Una de las dos playas que están separadas en e...,5.0,parque,google_maps,es,Una de las dos playas que están separadas en e...,😊,positivo
2,Playa Manuel Antonio,Plage exceptionnelle accessible uniquement en ...,5.0,parque,google_maps,fr,Playa excepcional a la que sólo se puede acced...,NaN,positivo


## 2. Cargar modelo de embeddings

In [3]:
modelo_emb = cargar_modelo_embeddings()

Cargando modelo de embeddings: intfloat/multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5197.01it/s]


## 3. Comparación de estrategias de chunking

In [4]:
# Estrategia 1: por reseña completa
chunks_resena = chunking_por_resena(corpus)
print(f'Chunks por reseña   : {len(chunks_resena)}')
print(f'Ejemplo: {chunks_resena[0]["texto"][:100]}...')
print()

Chunks por reseña   : 5057
Ejemplo: La playa de Manuel Antonio es absolutamente impresionante. Las olas estaban dando un espectáculo, lo...



In [5]:
# Estrategia 2: por oración
chunks_oracion = chunking_por_oracion(corpus)
print(f'Chunks por oración  : {len(chunks_oracion)}')
print(f'Ejemplo: {chunks_oracion[0]["texto"][:100]}')
print()

print('=== COMPARACIÓN ===')
print(f'Por reseña : {len(chunks_resena):,} chunks — contexto completo por reseña')
print(f'Por oración: {len(chunks_oracion):,} chunks — más granular, mayor precisión')

Chunks por oración  : 16035
Ejemplo: La playa de Manuel Antonio es absolutamente impresionante

=== COMPARACIÓN ===
Por reseña : 5,057 chunks — contexto completo por reseña
Por oración: 16,035 chunks — más granular, mayor precisión


## 4. Generar embeddings y construir índice FAISS
Usamos la estrategia **por reseña** para el índice principal (mejor para respuestas contextuales).

In [6]:
# Generar embeddings (se cachean en disco automáticamente)
embeddings = generar_embeddings(chunks_resena, modelo_emb)
print(f'Shape embeddings: {embeddings.shape}')

Cargando embeddings desde cache: ../../data/processed/embeddings_cache.pkl
Shape embeddings: (5057, 384)


In [7]:
# Construir índice FAISS
indice = construir_indice_faiss(embeddings)
print(f'Total vectores en índice: {indice.ntotal}')

# Guardar chunks para recuperar metadatos
guardar_chunks(chunks_resena)

Cargando índice FAISS desde: ../../data/processed/faiss_index.bin
Total vectores en índice: 5057
Chunks guardados en: ../../data/processed/chunks_cache.pkl


## 5. Pruebas de búsqueda semántica

In [9]:
# Prueba de búsqueda sin filtros
pregunta = '¿Qué restaurantes ofrecen un buen servicio?'
resultados = buscar_chunks(pregunta, modelo_emb, indice, chunks_resena, top_k=3)

print(f'Pregunta: {pregunta}\n')
for i, r in enumerate(resultados, 1):
    print(f'[{i}] {r["lugar"]} ({r["categoria"]}) | {r["calificacion"]}⭐ | Score: {r["score"]:.4f}')
    print(f'    {r["texto"][:120]}...')
    print()

Pregunta: ¿Qué restaurantes ofrecen un buen servicio?

[1] Restaurante Mi Finca (restaurante) | 4.0⭐ | Score: 0.8788
    Buen servicio y buena comida. Buena parada para degustar comida local....

[2] Restaurante Fortuneño (restaurante) | 4.0⭐ | Score: 0.8717
    Muy buena comida, variada y bien presentada con un servicio excelente, en un ambiente algo ruidoso....

[3] Hotel Roca Negra del Arenal (alojamiento) | 5.0⭐ | Score: 0.8712
    El personal es amable y siempre está dispuesto a ayudar con la reserva de excursiones en la zona.
Recomendé algunos rest...



In [10]:
# Prueba con filtro de categoría
pregunta = '¿Qué restaurantes recomiendan?'
resultados = buscar_chunks(pregunta, modelo_emb, indice, chunks_resena,
                           top_k=3, filtro_categoria='restaurante')

print(f'Pregunta: {pregunta} (filtro: restaurante)\n')
for i, r in enumerate(resultados, 1):
    print(f'[{i}] {r["lugar"]} | {r["calificacion"]}⭐ | Score: {r["score"]:.4f}')
    print(f'    {r["texto"][:120]}...')
    print()

Pregunta: ¿Qué restaurantes recomiendan? (filtro: restaurante)

[1] Soda Brasilito | 5.0⭐ | Score: 0.8743
    Comida deliciosa, buen precio, y con un gran ambiente. Los dueños del restaurante son muy amables y atentos. Muy recomen...

[2] Restaurante Fortuneño | 5.0⭐ | Score: 0.8708
    Excelente comida. Personal muy amable. Los platos estaban muy bien preparados. Buena relación precio y prestaciones. Sól...

[3] Azul Ocean Club Hotel | nan⭐ | Score: 0.8699
    súper recomendado! el lugar súper bonito, la atención del personal estuvo maravillosa ( Ariana y los muchachos del resta...



In [11]:
# Prueba con filtro de polaridad
pregunta = '¿Algún lugar con malas experiencias?'
resultados = buscar_chunks(pregunta, modelo_emb, indice, chunks_resena,
                           top_k=3, filtro_polaridad='negativo')

print(f'Pregunta: {pregunta} (filtro: negativo)\n')
for i, r in enumerate(resultados, 1):
    print(f'[{i}] {r["lugar"]} | {r["calificacion"]}⭐ | Score: {r["score"]:.4f}')
    print(f'    {r["texto"][:120]}...')
    print()

Pregunta: ¿Algún lugar con malas experiencias? (filtro: negativo)



## 6. Generación de respuestas con Gemini

In [ ]:
import os

API_KEY = os.getenv('GEMINI_API_KEY', 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
modelo_gemini = configurar_gemini(API_KEY)

In [13]:
# Prueba completa del pipeline RAG
preguntas_prueba = [
    '¿Qué lugares de naturaleza recomiendan en Costa Rica?',
    '¿Cuál es el mejor restaurante del corpus?',
    '¿Hay algún lugar con malas reseñas de servicio?',
    '¿Qué dicen de Corcovado National Park?',
]

historial = []

for pregunta in preguntas_prueba:
    print(f'Usuario: {pregunta}')
    respuesta, chunks = pipeline_rag(
        pregunta, modelo_emb, indice, chunks_resena, modelo_gemini, historial
    )
    print(f'TuriGuía: {respuesta}')
    print(f'(Basado en {len(chunks)} reseñas recuperadas)')
    print('-' * 60)

    # Actualizar historial
    historial.append({'role': 'user', 'content': pregunta})
    historial.append({'role': 'assistant', 'content': respuesta})

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Usuario: ¿Qué lugares de naturaleza recomiendan en Costa Rica?
TuriGuía: ¡Hola! Qué alegría saludarte. Como TuriGuía, me encanta ayudarte a planificar tu viaje basándome en las experiencias reales de otros viajeros. En Costa Rica la naturaleza es simplemente espectacular, y según las reseñas de quienes ya han estado allí, hay tres lugares imperdibles que deberías considerar:

1. **La Paz Waterfall Gardens Nature Park (parque):** 
   Este increíble parque privado está en las tierras altas centrales, a solo 45 minutos del aeropuerto de San José. Los viajeros destacan que es como "entrar a un sueño tropical". Cuenta con exhibiciones de aves, insectos, reptiles y hermosos jardines de orquídeas. Además, tiene senderos en la jungla a lo largo de un río que te llevan a ver cinco cascadas impresionantes (terminando en las famosas Cataratas de La Paz). 
   *Un tip de los viajeros:* El sendero de las cascadas tiene cientos de escaleras empinadas y resbaladizas, por lo que se recomienda llevar bo

## 7. Comparación RAG vs sin RAG

In [14]:
from src.modelos.rag_utils import construir_prompt, generar_respuesta

pregunta_test = '¿Qué opina la gente de Monteverde Country Lodge?'

# Sin RAG: solo la pregunta al modelo
prompt_sin_rag = f'Responde esta pregunta sobre turismo en Costa Rica: {pregunta_test}'
respuesta_sin_rag = generar_respuesta(prompt_sin_rag, modelo_gemini)

# Con RAG: con contexto del corpus
respuesta_con_rag, chunks = pipeline_rag(
    pregunta_test, modelo_emb, indice, chunks_resena, modelo_gemini, []
)

print('=== SIN RAG ===')
print(respuesta_sin_rag)
print()
print('=== CON RAG ===')
print(respuesta_con_rag)
print()
print(f'Chunks utilizados: {len(chunks)}')

=== SIN RAG ===
**Monteverde Country Lodge** es una de las opciones de hospedaje de rango medio más conocidas y tradicionales en la zona del bosque nuboso de Monteverde, Costa Rica. En plataformas de opinión como TripAdvisor, Booking.com y Google Reviews, el hotel suele tener calificaciones muy buenas (generalmente entre **4 y 4.5 de 5 estrellas**).

Aquí te presento un resumen de las opiniones de los huéspedes, dividido entre lo más positivo y los puntos de mejora:

### Lo que más le gusta a la gente (Puntos Fuertes):

1. **Ambiente rústico y acogedor:** Los huéspedes coinciden en que el hotel tiene un diseño de cabaña de montaña muy acogedor, con mucha madera y rodeado de hermosos jardines donde es común ver aves (como colibríes) y vegetación típica del bosque nuboso.
2. **Excelente servicio y hospitalidad:** La atención del personal es uno de los puntos más altos. Los visitantes suelen destacar la amabilidad de los recepcionistas y del personal del restaurante, quienes siempre están